# Multi level Perceptron (MLP): Using Numpy and Pandas

* Develop and train a MLP model to accurately classify MNIST data. 
* Use one hidden layer, one input layer and one output layer. 
* Test and choose an appropriate activation function, Optimizer and loss function. 
* Implement forward-propagation and Back-propagation.

In [1]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt

## Loading MNIST data
Load data and mention data properties

In [2]:
# Loading test and train data
train_data_input = pd.read_csv('MNIST_CSV/mnist_train.csv', header=None)
test_data_input = pd.read_csv('MNIST_CSV/mnist_test.csv', header=None)

In [3]:
# Data properties

print('Data shape : ', train_data_input.shape)

print('Image length : ', train_data_input.iloc[1,1:].shape)

print('Feature Data type : ', type(train_data_input.iloc[1,1]))

print('Class Data type : ', type(train_data_input.iloc[0,0]))


Data shape :  (60000, 785)
Image length :  (784,)
Feature Data type :  <class 'numpy.int64'>
Class Data type :  <class 'numpy.int64'>


In [4]:
# Separating Class and Normalizing data from -0.5 to 0.5

train_data = (train_data_input.iloc[:,1:] / 255).to_numpy() - 0.5

y_train = train_data_input.iloc[:,0].to_numpy()

test_data = (test_data_input.iloc[:,1:]  / 255).to_numpy() - 0.5

y_test = test_data_input.iloc[:,0].to_numpy()

print(f'Minimum and Maximum feature values: {train_data.min()}, {train_data.max()}')

Minimum and Maximum feature values: -0.5, 0.5


# Defining parameters

Initializing parameters and storing them in a dictionary

In [5]:
# Fixing seed
np.random.seed(42)

# Initializing weight for Input layer
fc1_weights = np.random.randn(784, 256) * np.sqrt(2/784)

# Initializing bias for Input layer
fc1_bias = np.ones((1, 256)) * 0.01

# Initializing weight for FC1 layer
fc2_weights = np.random.randn   (256,128) * np.sqrt(2/256)

#Initializing bias for FC1 layer
fc2_bias = np.ones((1,128)) *.01

# Initializing weight for FC2 layer
output_weights = np.random.randn(128,10) * np.sqrt(2/128)

#Initializing bias for FC2 layer
output_bias = np.ones((1,10)) * 0.01

# Defining parameters dictionary
param ={
    'fc1_weights' : fc1_weights,
    'fc1_bias' : fc1_bias,
    'fc2_weights' : fc2_weights,
    'fc2_bias' : fc2_bias,
    'output_weights' : output_weights,
    'output_bias' : output_bias
}

## Building the FC1 layer

* Building the first fully connected layer of MPL. Using Weight and bias matrices to expand
* Initializing weight with random values to break neuron symmetry



In [6]:


# Input layer function
def forward_fc1(input_array = None, weights = param['fc1_weights'], bias = param['fc1_bias']):

    output_matrix = np.matmul(input_array , weights) + bias

    return output_matrix

fc1_output = forward_fc1 (train_data)


# Adding activation layer

Introducing non linearity in neurons through hidden layer to make model functionally strong. Using ReLU, Sigmoid and Tanh as activation functions. Compressing the layer to 512 neurons.

In [7]:
# Creating a function for activation layer

# Activation layer function 
def Layer_activation(input_matrix = None, activation = 'ReLU'):

    if activation == 'ReLU':

        output_matrix = np.maximum(0, input_matrix)
    
        return output_matrix
    
    if activation == 'Leaky ReLU':

        output_matrix = np.maximum(0.01 * input_matrix, input_matrix)
    
        return output_matrix

Activation_output =  Layer_activation (fc1_output)


# Building FC2 layer

* Building a second hidden fully connected layer to ensure gradual compression of neurons.
* Initializing weight with random values to break neuron symmetry


In [8]:
# Layer_2 is a fully connected perceptron layer consisting of 256 neurons

# Fc2 layer function
def forward_fc2 (input_matrix = None, weights = param['fc2_weights'], bias = param['fc2_bias']):
    
    output_matrix = np.matmul( input_matrix , weights) + bias

    return output_matrix

fc2_output = forward_fc2 (Activation_output)



# Activation layer

Introducing another non linearity in neurons through hidden layer to make model functionally strong. Using ReLU, Sigmoid and Tanh as activation functions. 

Z3 = ReLU( ReLU( XW1 ) W2 ) W3

In [9]:
# Creating a function for activation layer

Activation_output_2 =  Layer_activation (fc2_output)

# Output Layer

Output layer compress the FC2 output further into 10 classes (digits)

In [10]:

# Output layer function 
def forward_output (input_matrix = None, activation = 'softmax', weights = param['output_weights'], bias = param['output_bias']):
    
    output_matrix = np.matmul( input_matrix , weights ) + bias    

    if activation == 'sigmoid': 

        output_matrix = 1 / ( 1 + np.exp(-output_matrix) )

    if activation == 'softmax':

        z = output_matrix - output_matrix.max( axis=1, keepdims = True)

        exp_z = np.exp(z)

        output_matrix = exp_z / exp_z.sum( axis=1, keepdims = True)

    return output_matrix

predictions = forward_output(
    Activation_output_2,
    'softmax',
    param['output_weights'],
    param['output_bias']
)


# One hot encoding for target variables for all training data
Converting the target array into 2d one hot encoded array for easier loss calculation 

In [11]:
One_hot_encoded_train = np.zeros((train_data.shape[0],10))

One_hot_encoded_train[np.arange(train_data.shape[0]),y_train] =1

print(One_hot_encoded_train)

[[0. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 1. 0.]]


# Variance analysis

Variance should neither shrink or increase by order of a magnitude through each layer.

In [12]:
# Comparing variance

print('Input variance: ', np.mean(np.var(train_data, axis=0)))
print('FC1 Layer variance: ', np.mean(np.var(fc1_output, axis=0)))
print('Activation Layer 1 variance: ', np.mean(np.var(Activation_output, axis=0)))
print('FC2 Layer variance: ', np.mean(np.var(fc2_output, axis=0)))
print('Activation Layer variance: ', np.mean(np.var(Activation_output_2, axis=0)))      



Input variance:  0.06725132078460057
FC1 Layer variance:  0.13573078085209025
Activation Layer 1 variance:  0.05618887083549106
FC2 Layer variance:  0.10894945535543159
Activation Layer variance:  0.0420532358939949


## Conclusions of variance analysis

* ReLU activation layer only activated when input range was changed from (0,1) to (0.5,0.5)
* ReLU Reduces variance to 0 when bias is zero. Bias needs to be small positive value
* Leaky ReLU performs better is preventing the variance from turning to 0
* Zero centered input and initial weights are essential for variance to not collapse in the activation (ReLU) layer

# Loss function
Using cross entropy as loss function for simple loss gradient function.
* Loss in cross entropy is always positive number as predictions range from (0, 1)

In [13]:
# Implementing cross entropy loss function

def cross_entropy (input_array = None , target = None):

    epsilon = 1e-8

    loss = -np.sum(target * np.log(input_array + epsilon), axis=1) 
    
    return loss

loss_initial = cross_entropy(predictions, One_hot_encoded_train)

# Mean loss and accuracy
print( np.mean(loss_initial))


2.4619276262896173


# Gradient calculations




## Backpropagation — Gradients of Weights and Biases w.r.t Loss

Gradient is defined as the **rate of change of loss with respect to a variable**. Using the chain rule, gradients are propagated in reverse through the network: **Output Layer → FC2 → FC1**.

Normalizing gradients of weights and biases to ensure gradient doesn't get impacted by large sample size. 
- `dz` = dz / sample size

**Notation:**
- `z` = pre-activation output
- `a` = post-activation output (after ReLU)
- `dz` = dL/dz &nbsp;|&nbsp; `dw` = dL/dW &nbsp;|&nbsp; `db` = dL/db &nbsp;|&nbsp; `da` = dL/da

In [14]:
gradient_params = {
    'dw3' : None,
    'dw2' : None,
    'dw1' : None,
    'db3' : None,
    'db2' : None,
    'db1' : None
}

In [15]:

def gradient (fc1_output = None, Activation_output = None, fc2_output = None, Activation_output_2 = None, predictions = None):

    w = train_data.shape[0]
    
    # --------------------------------------------------------------
    # OUTPUT LAYER (FC3)
    # Loss: cross-entropy | Activation: softmax
    # dL/dz3 = predictions - y  (softmax + cross-entropy combined)
    # --------------------------------------------------------------

    # Gradient of loss w.r.t. output layer pre-activation
    dz3 = predictions - One_hot_encoded_train


    # Gradient of loss w.r.t. output layer weights
    # dL/dW3 = A2^T @ dz3
    dw3 = np.transpose(Activation_output_2) @ dz3

    # Normalizing gradients
    dw3 /= w

    # Storing gradient in dictionary
    gradient_params['dw3'] = dw3

    # Gradient of loss w.r.t. output layer bias
    db3 = np.sum(dz3, axis=0, keepdims=True)

    # Normalizing gradients
    db3 /= w

    # Storing gradient in dictionary
    gradient_params['db3'] = db3


    # --------------------------------------------------------------
    # SECOND HIDDEN LAYER (FC2)
    # Activation: ReLU  →  ReLU gradient = 1 if z > 0, else 0
    # --------------------------------------------------------------

    # Gradient of loss w.r.t. FC2 post-activation (backdrop through output weights)
    # dL/dA2 = dz3 @ W3^T
    da2 = dz3 @ np.transpose(param['output_weights'])

    # Gradient of loss w.r.t. FC2 pre-activation (backprop through ReLU)
    # ReLU derivative: pass gradient only where FC2 output was positive
    dz2 = da2 * (fc2_output > 0)

    # Gradient of loss w.r.t. FC2 weights
    # dL/dW2 = A1^T @ dz2
    dw2 = np.transpose(Activation_output) @ dz2

    # Normalizing gradients
    dw2 /= w

    # Storing gradient in dictionary
    gradient_params['dw2'] = dw2

    # Gradient of loss w.r.t. FC2 bias
    db2 = np.sum(dz2, axis=0, keepdims=True)

    # Normalizing gradients
    db2 /= w

    # Storing gradient in dictionary
    gradient_params['db2'] = db2

    # --------------------------------------------------------------
    # FIRST HIDDEN LAYER (FC1)
    # Activation: ReLU  →  ReLU gradient = 1 if z > 0, else 0
    # --------------------------------------------------------------

    # Gradient of loss w.r.t. FC1 post-activation (backprop through FC2 weights)
    # dL/dA1 = dz2 @ W2^T
    da1 = dz2 @ np.transpose(param['fc2_weights'])

    # Gradient of loss w.r.t. FC1 pre-activation (backprop through ReLU)
    # ReLU derivative: pass gradient only where FC1 output was positive
    dz1 = da1 * (fc1_output > 0)

    # Gradient of loss w.r.t. FC1 weights
    # dL/dW1 = X^T @ dz1
    dw1 = np.transpose(train_data) @ dz1

    # Normalizing gradients
    dw1 /= w

    # Storing gradient in dictionary
    gradient_params['dw1'] = dw1

    # Gradient of loss w.r.t. FC1 bias
    db1 = np.sum(dz1, axis=0, keepdims=True)

    # Normalizing gradients
    db1 /= w

    # Storing gradient in dictionary
    gradient_params['db1'] = db1



# Gradient decent

Changing the weights according to the gradients and comparing loss function.

In [16]:
    # learning rate is a hyperparameter to control rate of change of step size
    
def gradient_decent (learning_rate = 0.01):

    # --------------------------------------------------------------
    # WEIGHTS GRADIENT DECENT
    # STEP SIZE = learning_rate * Gradient
    # --------------------------------------------------------------

    param['output_weights'] -= gradient_params['dw3'] * learning_rate

    param['fc2_weights'] -= gradient_params['dw2'] * learning_rate

    param['fc1_weights'] -= gradient_params['dw1'] * learning_rate

    # --------------------------------------------------------------
    # BIASES GRADIENT DECENT
    # STEP SIZE = learning_rate * Gradient
    # --------------------------------------------------------------

    param['output_bias'] -= gradient_params['db3'] * learning_rate

    param['fc2_bias'] -= gradient_params['db2'] * learning_rate

    param['fc1_bias'] -= gradient_params['db1'] * learning_rate


In [17]:
# Calculating and comparing Loss

def loss_calculation ():

    fc1_output = forward_fc1 (train_data, param['fc1_weights'], param['fc1_bias'] )

    Activation_output_1 =  Layer_activation (fc1_output)

    fc2_output = forward_fc2 (Activation_output_1, param['fc2_weights'], param['fc2_bias'])

    Activation_output_2 =  Layer_activation (fc2_output)

    predictions = forward_output (Activation_output_2, 'softmax', param['output_weights'], param['output_bias'])

    loss = cross_entropy( predictions, One_hot_encoded_train)

    return loss

# Mean Loss and Accuracy comparison

Loss = np.mean(loss_calculation())

print(Loss)


2.4619276262896173


# Implementing multiple passes

Performing multiple epochs to reduce loss

In [18]:
def train_step (learning_rate = 0.01):

    fc1_output = forward_fc1 (train_data, param['fc1_weights'], param['fc1_bias'])

    Activation_output =  Layer_activation (fc1_output)

    fc2_output = forward_fc2 (Activation_output, param['fc2_weights'], param['fc2_bias'])

    Activation_output_2 =  Layer_activation (fc2_output)

    predictions = forward_output (Activation_output_2)
    
    gradient( fc1_output, Activation_output, fc2_output, Activation_output_2, predictions)

    gradient_decent (learning_rate)

    loss = cross_entropy( predictions, One_hot_encoded_train)

    prediction_labels = np.argmax(predictions, axis = 1 )

    print (f'Iteration: {epoch}',f'Mean loss: {np.mean(loss)}', f'Accuracy : {np.mean( prediction_labels == y_train)}')

epochs = 200

for epoch in range(epochs):
    if epoch < 7:
        learning_rate = 0.09

    elif epoch < 12:
        learning_rate = 0.08
        
    elif epoch < 20:
        learning_rate = 0.075

    elif epoch < 30:
        learning_rate = 0.065

    elif epoch < 40:
        learning_rate = 0.06

    elif epoch < 170:
        learning_rate = 0.053

    else:
        learning_rate = 0.05


    train_step (learning_rate)
    

Iteration: 0 Mean loss: 2.4619276262896173 Accuracy : 0.0798
Iteration: 1 Mean loss: 2.2157088966936413 Accuracy : 0.23521666666666666
Iteration: 2 Mean loss: 2.096096325384845 Accuracy : 0.3236
Iteration: 3 Mean loss: 1.993456308154959 Accuracy : 0.43033333333333335
Iteration: 4 Mean loss: 1.89912522733087 Accuracy : 0.46958333333333335
Iteration: 5 Mean loss: 1.81180838402351 Accuracy : 0.5382666666666667
Iteration: 6 Mean loss: 1.7327941278265944 Accuracy : 0.5439
Iteration: 7 Mean loss: 1.668876862138463 Accuracy : 0.58835
Iteration: 8 Mean loss: 1.6314547505715609 Accuracy : 0.55215
Iteration: 9 Mean loss: 1.602037482369404 Accuracy : 0.5716
Iteration: 10 Mean loss: 1.6276069861297544 Accuracy : 0.5111833333333333
Iteration: 11 Mean loss: 1.5721663938842976 Accuracy : 0.5449333333333334
Iteration: 12 Mean loss: 1.5575556393614287 Accuracy : 0.5391166666666667
Iteration: 13 Mean loss: 1.3947544889965025 Accuracy : 0.6465166666666666
Iteration: 14 Mean loss: 1.3187800122928224 Accur